# 01 · Data collection — international match results

Downloads international results and penalty shootouts from the
[martj42/international_results](https://github.com/martj42/international_results) dataset,
keeps matches from **1 Jan 2024 to 4 Jun 2026** (the window used for modelling), and adds outcome columns.

**Outputs** (written to `data/`):
- `results_clean.csv` — one row per match. This is the base table that `matches_stats.csv` was built on (see the README).
- `shootouts_clean.csv` — penalty shootouts in the same window.

In [1]:
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/martj42/international_results/master"
START_DATE, END_DATE = "2024-01-01", "2026-06-04"   # last date in the modelling data

## 1. Download

In [2]:
results   = pd.read_csv(StringIO(requests.get(f"{BASE_URL}/results.csv", timeout=60).text))
shootouts = pd.read_csv(StringIO(requests.get(f"{BASE_URL}/shootouts.csv", timeout=60).text))

results["date"]   = pd.to_datetime(results["date"])
shootouts["date"] = pd.to_datetime(shootouts["date"])
print(f"Results: {len(results):,} rows | Shootouts: {len(shootouts):,} rows")

Results: 49,547 rows | Shootouts: 683 rows


## 2. Filter to the modelling window and flag competitive matches
Recent matches only: team strength changes over time, so older results add noise. Friendlies are kept but flagged with `is_competitive = False`.

In [3]:
in_window = lambda df: df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)].copy()
results = in_window(results)
print(f"Matches {START_DATE} → {END_DATE}: {len(results):,}")

COMPETITIVE = {
    "FIFA World Cup", "FIFA World Cup qualification",
    "UEFA Euro", "UEFA Euro qualification", "UEFA Nations League",
    "Copa América", "CONMEBOL–UEFA Cup of Champions",
    "Africa Cup of Nations", "Africa Cup of Nations qualification",
    "AFC Asian Cup", "AFC Asian Cup qualification",
    "CONCACAF Gold Cup", "CONCACAF Nations League",
}
results["is_competitive"] = results["tournament"].isin(COMPETITIVE)

Matches 2024-01-01 → 2026-06-04: 2,464


## 3. Outcome columns

In [4]:
home_win = results["home_score"] > results["away_score"]
away_win = results["home_score"] < results["away_score"]

results["home_or_away_win"] = np.select([home_win, away_win], ["home", "away"], "draw")   # model target
results["winner"]           = np.select([home_win, away_win],
                                        [results["home_team"], results["away_team"]], "Draw")
results["goal_diff"]   = results["home_score"] - results["away_score"]
results["total_goals"] = results["home_score"] + results["away_score"]

## 4. Merge penalty shootouts
In `shootouts.csv`, `winner` is the team that won the shootout. For those matches the 90-minute result is a draw, so the shootout winner replaces `winner`.

In [5]:
results = results.merge(
    shootouts[["date", "home_team", "away_team", "winner"]].rename(columns={"winner": "shootout_winner"}),
    on=["date", "home_team", "away_team"], how="left",
)
results["went_to_penalties"] = results["shootout_winner"].notna()
results["winner"] = np.where(results["went_to_penalties"], results["shootout_winner"], results["winner"])

shootouts["loser"] = np.where(shootouts["winner"] == shootouts["home_team"],
                              shootouts["away_team"], shootouts["home_team"])
shootouts["winner_side"] = np.where(shootouts["winner"] == shootouts["home_team"], "home", "away")
shootouts = in_window(shootouts)
print(f"Matches that went to penalties: {results['went_to_penalties'].sum()}")

Matches that went to penalties: 53


## 5. Match-level indicator columns

In [6]:
results["home_win"]  = (results["home_or_away_win"] == "home").astype(int)
results["away_win"]  = (results["home_or_away_win"] == "away").astype(int)
results["is_draw"]   = (results["home_or_away_win"] == "draw").astype(int)
results["home_clean_sheet"] = (results["away_score"] == 0).astype(int)
results["away_clean_sheet"] = (results["home_score"] == 0).astype(int)
results["over_2_5"]  = (results["total_goals"] > 2).astype(int)
results["btts"]      = ((results["home_score"] > 0) & (results["away_score"] > 0)).astype(int)  # both teams scored

## 6. Save

In [7]:
COL_ORDER = [
    "date", "tournament", "is_competitive", "city", "country", "neutral",
    "home_team", "away_team", "home_score", "away_score", "goal_diff", "total_goals",
    "home_or_away_win", "winner", "went_to_penalties", "shootout_winner",
    "home_win", "away_win", "is_draw", "home_clean_sheet", "away_clean_sheet", "over_2_5", "btts",
]
results[COL_ORDER].to_csv(DATA / "results_clean.csv", index=False)
shootouts[["date", "home_team", "away_team", "winner", "loser", "winner_side"]].to_csv(
    DATA / "shootouts_clean.csv", index=False)

print(f"results_clean.csv:   {len(results):,} rows")
print(f"shootouts_clean.csv: {len(shootouts):,} rows")
print(results["home_or_away_win"].value_counts().to_string())

results_clean.csv:   2,464 rows
shootouts_clean.csv: 53 rows
home_or_away_win
home    1166
away     717
draw     581
